# Word Predictor — GPT-2 + WikiText-2

Fine-tune GPT-2 small on WikiText-2, then test word prediction with beam search.

**Select GPU runtime**: `Runtime > Change runtime type > T4 GPU`

In [ ]:
# ============================================================
# 1. SETUP
# ============================================================
!pip install -q torch transformers datasets tqdm

import torch
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# 2. LOAD & TOKENIZE WIKITEXT-2
# ============================================================
import os, math
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)

MODEL_NAME = 'gpt2'
OUTPUT_DIR = 'models/gpt2-wikitext2'
BLOCK_SIZE = 256

print('Loading WikiText-2 ...')
raw = load_dataset('wikitext', 'wikitext-2-raw-v1')
print(f"Train: {len(raw['train']):,}  Val: {len(raw['validation']):,}  Test: {len(raw['test']):,}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(ex):
    return tokenizer(ex['text'], truncation=False, return_attention_mask=False)

tokenized = raw.map(tokenize_fn, batched=True, remove_columns=['text'], desc='Tokenizing')

def group_texts(ex):
    concat = {k: sum(ex[k], []) for k in ex}
    n = (len(concat['input_ids']) // BLOCK_SIZE) * BLOCK_SIZE
    res = {k: [v[i:i+BLOCK_SIZE] for i in range(0, n, BLOCK_SIZE)] for k, v in concat.items()}
    res['labels'] = res['input_ids'].copy()
    return res

lm = tokenized.map(group_texts, batched=True, desc='Grouping')
print(f"Train blocks: {len(lm['train']):,}  Val blocks: {len(lm['validation']):,}")

In [ ]:
# ============================================================
# 3. TRAIN
# ============================================================
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    eval_strategy='steps',
    eval_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=lm['train'],
    eval_dataset=lm['validation'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('Training ...')
trainer.train()

ppl = math.exp(trainer.evaluate()['eval_loss'])
print(f'\nValidation perplexity: {ppl:.2f}')

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}')

In [ ]:
# ============================================================
# 4. TRANSFORMER PREDICTOR
# ============================================================
import torch
from typing import List, Tuple, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM

WORD_CHARS = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ-\'')

class TransformerPredictor:
    def __init__(self, model_path='gpt2', device=None, max_steps=10):
        self.device = device or (
            'cuda' if torch.cuda.is_available()
            else 'mps' if torch.backends.mps.is_available()
            else 'cpu'
        )
        print(f'[Predictor] Loading {model_path} on {self.device}')
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()
        self.max_steps = max_steps
        self.eos_id = self.tokenizer.eos_token_id

    def predict(self, context, prefix='', top_k=5, beam_width=20):
        if not context and not prefix:
            return []
        full = context
        if prefix:
            if full and not full.endswith(' '):
                full += ' '
            full += prefix
        ids = self.tokenizer.encode(full, return_tensors='pt').to(self.device)
        comps = self._beam(ids, prefix, beam_width)
        seen, out = set(), []
        for w, s in comps:
            k = w.lower()
            if k and k not in seen:
                seen.add(k)
                out.append((w, s))
        out.sort(key=lambda x: x[1], reverse=True)
        return [w for w, _ in out[:top_k]]

    def predict_from_text(self, typed, top_k=5):
        if not typed:
            return []
        if typed.endswith(' '):
            return self.predict(typed, '', top_k=top_k)
        parts = typed.rsplit(' ', 1)
        if len(parts) == 1:
            return self.predict('', parts[0], top_k=top_k)
        return self.predict(parts[0] + ' ', parts[1], top_k=top_k)

    def _beam(self, input_ids, prefix, bw):
        active = [(0.0, [])]
        done = []
        pl = prefix.lower()

        for _ in range(self.max_steps):
            if not active:
                break
            cands = []
            for lp, app in active:
                if app:
                    fids = torch.cat(
                        [input_ids, torch.tensor([app], device=self.device)],
                        dim=-1,
                    )
                else:
                    fids = input_ids

                with torch.no_grad():
                    logits = self.model(fids).logits[0, -1, :]
                log_p = torch.log_softmax(logits, dim=-1)
                vals, idxs = torch.topk(log_p, k=min(bw * 3, log_p.size(0)))

                for i in range(vals.size(0)):
                    tid = idxs[i].item()
                    nlp = lp + vals[i].item()
                    napp = app + [tid]

                    if tid == self.eos_id:
                        w = self._word(napp)
                        if w and w.lower().startswith(pl):
                            done.append((w, nlp))
                        continue

                    raw = self.tokenizer.decode(napp, skip_special_tokens=True)
                    text = raw.lstrip()

                    if not text:
                        cands.append((nlp, napp))
                        continue

                    if ' ' in text or '\n' in text or '\t' in text:
                        w = _clean(text.split()[0])
                        if w and w.lower().startswith(pl):
                            done.append((w, nlp))
                    else:
                        p = _clean(text)
                        if not pl or p.lower().startswith(pl):
                            cands.append((nlp, napp))

            cands.sort(key=lambda x: x[0], reverse=True)
            active = cands[:bw]

        for lp, app in active:
            w = self._word(app)
            if w and w.lower().startswith(pl):
                done.append((w, lp))
        return done

    def _word(self, ids):
        if not ids:
            return ''
        t = self.tokenizer.decode(ids, skip_special_tokens=True).lstrip()
        return _clean(t.split()[0]) if t.split() else _clean(t)


def _clean(t):
    o = []
    for c in t:
        if c in WORD_CHARS:
            o.append(c)
        else:
            break
    return ''.join(o)

In [ ]:
# ============================================================
# 5. TEST PREDICTIONS
# ============================================================
predictor = TransformerPredictor(OUTPUT_DIR)

tests = [
    'The quick brown ',
    'The quick br',
    'Machine learning is a ',
    'I went to the ',
    'pyt',
    'Artificial intelli',
    'The president of the ',
    'In the year ',
]

for t in tests:
    s = predictor.predict_from_text(t, top_k=5)
    print(f"  '{t}'  ->  {s}")

In [ ]:
# ============================================================
# 6. SAVE TO GOOGLE DRIVE (optional)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
import shutil
dst = '/content/drive/MyDrive/Wordprediktor/models/gpt2-wikitext2'
os.makedirs(dst, exist_ok=True)
shutil.copytree(OUTPUT_DIR, dst, dirs_exist_ok=True)
print(f'Saved -> {dst}')

In [ ]:
# ============================================================
# 7. INTERACTIVE
# ============================================================
while True:
    text = input('\nType (or quit): ')
    if text.lower() == 'quit':
        break
    print(f'  -> {predictor.predict_from_text(text, top_k=5)}')